In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf, col
from pyspark.sql.types import ArrayType, FloatType
import numpy as np

# Tạo phiên làm việc Spark
spark = SparkSession.builder \
    .appName("Collaborative Fuzzy Clustering") \
    .getOrCreate()

# Giả lập dữ liệu ảnh đa phổ
data = [
    (0, np.array([0.1, 0.2, 0.3]).tolist()),
    (1, np.array([0.4, 0.5, 0.6]).tolist()),
    (2, np.array([0.7, 0.8, 0.9]).tolist()),
    (3, np.array([0.2, 0.1, 0.4]).tolist())
]

# Tạo DataFrame từ dữ liệu
df = spark.createDataFrame(data, ["id", "features"])

# Định nghĩa các hàm tính toán
def fuzzy_membership(pixel, centers, m=2):
    pixel = np.array(pixel)
    centers = np.array(centers)
    distances = np.linalg.norm(centers - pixel, axis=1)
    distances = np.maximum(distances, 1e-10)  # Tránh chia cho 0
    u = 1 / (distances ** (2 / (m - 1)))
    u /= np.sum(u)
    return u.tolist()

def compute_centers(features, memberships, k):
    features = np.array(features)
    memberships = np.array(memberships)
    centers = []
    for i in range(k):
        membership = memberships[:, i]
        numerator = np.sum(membership[:, np.newaxis] * features, axis=0)
        denominator = np.sum(membership)
        centers.append(numerator / denominator)
    return centers

# UDF để tính toán độ tin cậy (membership) của pixel cho mỗi cụm
compute_memberships_udf = udf(lambda features, centers: fuzzy_membership(features, centers), ArrayType(FloatType()))

# Số cụm
k = 2
m = 2

# Giả lập các trọng tâm cụm ban đầu
initial_centers = np.array([[0.1, 0.2, 0.3], [0.5, 0.6, 0.7]])

# Tính toán độ tin cậy cho mỗi Pixel
df = df.withColumn("memberships", compute_memberships_udf(col("features"), col("features").cast(ArrayType(FloatType()))))

# Chạy nhiều vòng lặp để cập nhật trọng tâm và độ tin cậy
for _ in range(10):  # Số vòng lặp
    df_memberships = df.select("id", "memberships")
    memberships_rdd = df_memberships.rdd.map(lambda row: (row[0], row[1]))
    
    # Tính toán trọng tâm cụm mới
    features = np.array(df.select("features").rdd.map(lambda row: row[0]).collect())
    memberships = np.array([m[1] for m in memberships_rdd.collect()])
    new_centers = compute_centers(features, memberships, k)
    
    # Cập nhật trọng tâm cụm cho lần lặp tiếp theo
    initial_centers = np.array(new_centers)
    
    # Tính toán lại ma trận thành viên với trọng tâm cụm mới
    df = df.withColumn("memberships", compute_memberships_udf(col("features"), initial_centers.tolist()))

# Hiển thị kết quả
df.show()

# Dừng phiên làm việc với Spark
spark.stop()


24/08/24 07:08:04 WARN Utils: Your hostname, ubuntu resolves to a loopback address: 127.0.1.1; using 172.20.10.2 instead (on interface wlp6s0)
24/08/24 07:08:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/08/24 07:08:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/08/24 07:08:06 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/08/24 07:08:15 ERROR PythonRunner: Python worker exited unexpectedly (crashed)
org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/usr/lib/spark/python/lib/pyspark.zip/pyspark/worker.py", line 1225, in main
    eval_type = read_int(infile)
  File "/usr/lib/spark/python/lib/pyspark.zip/pyspark/serializers.py", line 596, in read_int
    raise EOFError
EOFEr

PythonException: 
  An exception was thrown from the Python worker. Please see the stack trace below.
Traceback (most recent call last):
  File "/tmp/ipykernel_468613/881452708.py", line 44, in <lambda>
  File "/tmp/ipykernel_468613/881452708.py", line 26, in fuzzy_membership
  File "/home/ubuntu/venv/lib/python3.10/site-packages/numpy/linalg/linalg.py", line 2583, in norm
    return sqrt(add.reduce(s, axis=axis, keepdims=keepdims))
numpy.exceptions.AxisError: axis 1 is out of bounds for array of dimension 1


In [3]:
df

DataFrame[id: bigint, features: array<double>, memberships: array<float>]